In [51]:
using MPSKit, TensorKit, Plots, LaTeXStrings, LinearAlgebra

In [52]:
d = 25 # local Hilbert space dimension, before truncating
D = 20

20

In [53]:
function matrix_elems(d)
    phi    = zeros(ComplexF64, (d, d))
    phi_sq = zeros(ComplexF64, (d, d))
    pi_sq  = zeros(ComplexF64, (d, d))
    phi_4  = zeros(ComplexF64, (d, d))

    # phi = (a + a^\dagger)/sqrt(2)
    for i in 2:d
    	val = sqrt((i - 1)/ 2)
        phi[i, i - 1] = val
        phi[i - 1, i] = val
    end

    # helper to fill phi_sq and pi_sq (same structure, sign flip on off-diagonals)
    function fill_quadratic!(M, sign)
        @inbounds for i in 1:d
            if i < d - 1
                val = sign * sqrt(i * (i + 1)) / 2
                M[i, i + 2] = val
                M[i + 2, i] = val  # Hermitian
            end
            M[i, i] = (2 * i - 1) / 2
        end
    end

    fill_quadratic!(phi_sq, +1)
    fill_quadratic!(pi_sq,  -1)

    # phi^4 in harmonic oscillator basis
    @inbounds for i in 1:d
        n = i - 1  # occupation number

        # diagonal
        phi_4[i, i] = (6 * n^2 + 6 * n + 3) / 4

        # connect |n> <-> |n+2>
        if i + 2 <= d
            val = (4 * n + 6) * sqrt((n + 1) * (n + 2)) / 4
            j = i + 2
            phi_4[i, j] = val
            phi_4[j, i] = val  # Hermitian
        end

        # connect |n> <-> |n+4>
        if i + 4 <= d
            val = sqrt((n + 1) * (n + 2) * (n + 3) * (n + 4)) / 4
            j = i + 4
            phi_4[i, j] = val
            phi_4[j, i] = val  # Hermitian
        end
    end
    return phi, phi_sq, pi_sq, phi_4
end

matrix_elems (generic function with 1 method)

In [54]:
phi, phi_sq, pi_sq, phi_4 = matrix_elems(d)

(ComplexF64[0.0 + 0.0im 0.7071067811865476 + 0.0im … 0.0 + 0.0im 0.0 + 0.0im; 0.7071067811865476 + 0.0im 0.0 + 0.0im … 0.0 + 0.0im 0.0 + 0.0im; … ; 0.0 + 0.0im 0.0 + 0.0im … 0.0 + 0.0im 3.4641016151377544 + 0.0im; 0.0 + 0.0im 0.0 + 0.0im … 3.4641016151377544 + 0.0im 0.0 + 0.0im], ComplexF64[0.5 + 0.0im 0.0 + 0.0im … 0.0 + 0.0im 0.0 + 0.0im; 0.0 + 0.0im 1.5 + 0.0im … 0.0 + 0.0im 0.0 + 0.0im; … ; 0.0 + 0.0im 0.0 + 0.0im … 23.5 + 0.0im 0.0 + 0.0im; 0.0 + 0.0im 0.0 + 0.0im … 0.0 + 0.0im 24.5 + 0.0im], ComplexF64[0.5 + 0.0im 0.0 + 0.0im … 0.0 + 0.0im 0.0 + 0.0im; 0.0 + 0.0im 1.5 + 0.0im … 0.0 + 0.0im 0.0 + 0.0im; … ; 0.0 + 0.0im 0.0 + 0.0im … 23.5 + 0.0im 0.0 + 0.0im; 0.0 + 0.0im 0.0 + 0.0im … 0.0 + 0.0im 24.5 + 0.0im], ComplexF64[0.75 + 0.0im 0.0 + 0.0im … 0.0 + 0.0im 0.0 + 0.0im; 0.0 + 0.0im 3.75 + 0.0im … 0.0 + 0.0im 0.0 + 0.0im; … ; 0.0 + 0.0im 0.0 + 0.0im … 828.75 + 0.0im 0.0 + 0.0im; 0.0 + 0.0im 0.0 + 0.0im … 0.0 + 0.0im 900.75 + 0.0im])

In [55]:
β = 1.0
μ = 0.5
κ = 0.1
m = 1

1

In [56]:
iden_mat = Array(I, d, d);
ham_mat = (m^2 * phi_sq + pi_sq)/2 + μ^2 * (iden_mat - cos(β * phi));

In [57]:
ϕ = TensorMap(phi, ℂ^d ← ℂ^d)
iden = TensorMap(iden_mat, ℂ^d ← ℂ^d)
# don't actually need these.
ϕ2 = TensorMap(phi_sq, ℂ^d ← ℂ^d)
π2 = TensorMap(pi_sq, ℂ^d ← ℂ^d)
ϕ4 = TensorMap(phi_4, ℂ^d ← ℂ^d)

25←25 TensorMap{ComplexF64, ComplexSpace, 1, 1, Vector{ComplexF64}}:
 codomain: ⊗(ℂ^25)
 domain: ⊗(ℂ^25)


In [44]:
L = 100 # number of lattice sites
J₀ = 1.0
J₁ = 0.2

0.2

In [45]:
single_site_terms_ham = [i => (π2 + m^2 * ϕ2)/2 + μ^2 * (iden - cos(β * ϕ)) for i in 1:L]
# these two terms come from the gradient
single_site_terms_grad1 = [i => κ * ϕ2 / 2 for i in 1:L-1]
single_site_terms_grad2 = [i => κ * ϕ2 / 2 for i in 2:L]
# final term comes from the two site coupling in the gradient
two_site_terms_grad = [(i, i + 1) => -κ * ϕ ⊗ ϕ for i in 1:L - 1]
# source terms
source_terms_init = [i => J₀ * ϕ for i in 2:L-1]
source_terms_fin = [i => J₁ * ϕ for i in 2:L-1]; # final source terms

In [46]:
chain = fill(ℂ^d, L)
ham = FiniteMPOHamiltonian(chain, single_site_terms_grad1...,
    single_site_terms_grad2..., single_site_terms_ham...,
    two_site_terms_grad..., source_terms_init...);

In [47]:
ψ_gs = FiniteMPS(L, ℂ^d, ℂ^D)
ψ_gs, _, _ = find_groundstate(ψ_gs, ham, DMRG());

[ Info: DMRG init:	obj = +1.745946047606e+03	err = 9.9703e-01
[ Info: DMRG   1:	obj = +1.845412331597e+01	err = 5.1587958960e-02	time = 10.82 sec
[ Info: DMRG   2:	obj = +1.845412330211e+01	err = 2.0711018301e-06	time = 4.38 sec
[ Info: DMRG   3:	obj = +1.845412329990e+01	err = 1.1051029757e-06	time = 12.22 sec
[ Info: DMRG conv 4:	obj = +1.845412329990e+01	err = 3.8529939157e-11	time = 31.74 sec


In [26]:
expectation_value(ψ_gs, ham)

18.45412329147289 + 1.402254007174037e-15im

In [49]:
expectation_value(ψ_gs, ham)

18.454123299901106 + 2.3779700673612384e-15im